# 🇹🇭 Thai Constitution Pipeline — Kaggle / Colab Runner
## โครงการวิเคราะห์รัฐธรรมนูญไทย 38 ฉบับ (CPE232 Data Models)

---

| รายละเอียด | ค่า |
|-----------|-----|
| **Pipeline** | OCR (32 ฉบับ) + Text Extraction (6 ฉบับ) |
| **OCR Engine** | Typhoon OCR 1.5 (via API — ไม่ต้องใช้ GPU) |
| **Runtime แนะนำ** | CPU (GPU ไม่จำเป็น เพราะใช้ API call) |
| **เวลาโดยประมาณ** | 8–14 ชั่วโมง (ขึ้นอยู่กับจำนวนหน้า + rate limit) |

---

## 🗂️ โครงสร้าง Folder ที่จะได้หลัง Pipeline รัน

```
[WORK_DIR]/
├── ocr_cache/              ← Cache OCR ทีละหน้า (resume ได้ถ้า session ตาย)
│   ├── const_2475/
│   │   ├── page_001.md
│   │   ├── page_002.md
│   │   └── ...
│   └── const_2502/ ...
├── extracted_text/         ← ผลดึงข้อความ Text PDF
│   └── const_2550_raw.txt ...
├── processed/              ← ผลลัพธ์สุดท้าย (ดาวน์โหลดอันนี้)
│   ├── const_2475.json
│   ├── ... (38 ไฟล์)
│   ├── const_2564.json
│   ├── summary_ocr.csv
│   ├── summary_text_extraction.csv
│   └── qa_report.csv
└── logs/
    ├── ocr_pipeline.log
    └── text_extraction.log
```

---

## ⚙️ วิธีตั้งค่าต่อ Platform

### 🟠 Kaggle
1. ไปที่ **Notebook → Settings → Accelerator** → เลือก **None (CPU)** (ไม่ต้องใช้ GPU)
2. ไปที่ **Settings → Data** → เพิ่มชุดข้อมูล:
   - ชุดข้อมูล: `thai-constitutions-pdfs` (upload PDF ทั้ง 38 ไฟล์ไว้ที่นี่ ดูวิธีด้านล่าง)
3. ไปที่ **Settings → Secrets** → เพิ่ม:
   - Key: `TYPHOON_OCR_API_KEY` | Value: API Key จาก https://opentyphoon.ai
   - แล้วกด toggle ✅ ให้ Notebook นี้เข้าถึงได้
4. ไปที่ **Settings → Internet** → เปิด **Internet On** ✅ (จำเป็นสำหรับ API call)
5. รัน All Cells

### 🔵 Google Colab
1. เปิด Runtime → Change runtime type → เลือก **CPU**
2. ไปที่ 🔑 Secrets (ไอคอนกุญแจซ้ายมือ) → เพิ่ม:
   - Name: `TYPHOON_OCR_API_KEY` | Value: API Key
3. อัปโหลด PDF ทั้ง 38 ไฟล์ขึ้น Google Drive ใน folder `CPE232_Constitution/raw_pdfs/`
4. รัน All Cells (จะขอ Mount Drive ใน Cell 4)

### 💻 Local
1. ตั้งค่า `.env` ด้วย `TYPHOON_OCR_API_KEY=xxx` (มีอยู่แล้วใน `.env.example`)
2. ใช้ `python run_pipeline.py` ตรงๆ แทน Notebook นี้ได้เลย

---

## 📤 วิธี Upload PDF เป็น Kaggle Dataset

1. ไปที่ https://www.kaggle.com/datasets → **New Dataset**
2. ตั้งชื่อ: `thai-constitutions-pdfs`
3. อัปโหลด PDF ทั้ง 38 ไฟล์จาก `data/raw_pdfs/`
4. เลือก **Private** แล้ว Create
5. เมื่อสร้างแล้ว ไปที่ Notebook นี้ → Settings → Data → Add Dataset → ค้นหา `thai-constitutions-pdfs`

> ⚠️ **Kaggle Session Limit:** CPU session มีเวลา 9 ชั่วโมง/สัปดาห์ ถ้า pipeline ยังไม่เสร็จ
> ให้ดาวน์โหลด `/kaggle/working/ocr_cache/` ไปเก็บ แล้ว re-run — ระบบจะ Resume จาก cache ต่อ


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Environment Detection
# ═══════════════════════════════════════════════════════════════
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules
if not IS_COLAB:
    try:
        import google.colab
        IS_COLAB = True
    except ImportError:
        pass
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

ENV_NAME = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')

print(f"🖥️  Environment : {ENV_NAME}")
print(f"🐍  Python      : {sys.version.split()[0]}")

# ── GitHub repo ────────────────────────────────────────────────
GITHUB_ORG      = "Palapluem"
GITHUB_REPO     = "cpe232-datamodel-2025"
GITHUB_BRANCH   = "main"
SCRIPTS_SUBPATH = "project/data_preparation"
GITHUB_RAW_BASE = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_ORG}/{GITHUB_REPO}/{GITHUB_BRANCH}/{SCRIPTS_SUBPATH}"
)

print(f"\n📦  Source repo : https://github.com/{GITHUB_ORG}/{GITHUB_REPO}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Install Dependencies
# ═══════════════════════════════════════════════════════════════
import subprocess

print("📦 กำลังติดตั้ง Python packages...")

packages = [
    "typhoon-ocr>=0.2.0",
    "pymupdf>=1.24.0",
    "pdfplumber>=0.11.0",
    "pdf2image>=1.17.0",
    "Pillow>=10.0.0",
    "pythainlp>=5.0.0",
    "pandas>=2.0.0",
    "numpy>=1.26.0",
    "python-dotenv>=1.0.0",
    "tqdm>=4.66.0",
    "tenacity>=8.2.0",
    "jsonschema>=4.21.0",
    "requests>=2.31.0",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("⚠️  pip stderr:", result.stderr[-500:])
else:
    print("✅ Python packages ติดตั้งเสร็จ")

# ── Thai font for matplotlib ────────────────────────────────────
if IS_KAGGLE or IS_COLAB:
    print("\n🔤 ติดตั้งฟอนต์ภาษาไทย...")
    import urllib.request
    font_url  = "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf"
    font_path = Path("/tmp/Sarabun-Regular.ttf")
    try:
        urllib.request.urlretrieve(font_url, font_path)
        from matplotlib import font_manager
        font_manager.fontManager.addfont(str(font_path))
        print("✅ ฟอนต์ Sarabun ติดตั้งเสร็จ")
    except Exception as e:
        print(f"⚠️  ฟอนต์ไม่สามารถติดตั้งได้: {e} (กราฟอาจแสดงภาษาไทยผิดพลาด)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Path & Storage Setup
# ปรับ path ให้ถูกต้องตาม Environment
# ═══════════════════════════════════════════════════════════════

if IS_KAGGLE:
    # ─── Kaggle ─────────────────────────────────────────────────
    # Input:  /kaggle/input/thai-constitutions-pdfs/  ← Kaggle Dataset
    # Output: /kaggle/working/                        ← ดาวน์โหลดได้ท้าย session

    PDF_DIR     = Path("/kaggle/input/thai-constitutions-pdfs")
    WORK_DIR    = Path("/kaggle/working")
    SCRIPT_DIR  = WORK_DIR / "scripts"

elif IS_COLAB:
    # ─── Google Colab ───────────────────────────────────────────
    # PDF ต้องอยู่ใน Google Drive → ก่อน mount ใน Cell 4

    DRIVE_ROOT  = Path("/content/drive/MyDrive")
    PROJECT_DIR = DRIVE_ROOT / "CPE232_Constitution"   # ← เปลี่ยนถ้าต้องการ
    PDF_DIR     = PROJECT_DIR / "raw_pdfs"
    WORK_DIR    = PROJECT_DIR
    SCRIPT_DIR  = Path("/content/scripts")

else:
    # ─── Local ──────────────────────────────────────────────────
    # ใช้ structure ปกติของ repo
    BASE        = Path(".").resolve()
    PDF_DIR     = BASE / "data" / "raw_pdfs"
    WORK_DIR    = BASE
    SCRIPT_DIR  = BASE

# ── Derived paths ────────────────────────────────────────────────
OCR_CACHE_DIR = WORK_DIR / "ocr_cache"
TEXT_DIR      = WORK_DIR / "extracted_text"
PROCESSED_DIR = WORK_DIR / "processed"
LOG_DIR       = WORK_DIR / "logs"

# สร้าง directories
for d in [SCRIPT_DIR, OCR_CACHE_DIR, TEXT_DIR, PROCESSED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📁 Environment  : {ENV_NAME}")
print(f"📂 PDF_DIR      : {PDF_DIR}")
print(f"📂 WORK_DIR     : {WORK_DIR}")
print(f"📂 SCRIPT_DIR   : {SCRIPT_DIR}")
print(f"📂 OCR_CACHE    : {OCR_CACHE_DIR}")
print(f"📂 PROCESSED    : {PROCESSED_DIR}")

# ตรวจว่า PDF มีอยู่ไหม (Kaggle/Local)
if not IS_COLAB:  # Colab ยัง mount ไม่ได้ตอนนี้
    if PDF_DIR.exists():
        pdf_count = len(list(PDF_DIR.glob("*.pdf")))
        print(f"\n✅ พบ PDF {pdf_count}/38 ไฟล์ใน {PDF_DIR}")
        if pdf_count < 38:
            print(f"⚠️  พบ PDF แค่ {pdf_count} ไฟล์ (ต้องการ 38 ไฟล์)")
    else:
        print(f"\n❌ ไม่พบ PDF_DIR: {PDF_DIR}")
        if IS_KAGGLE:
            print("   → ไปที่ Notebook Settings → Data → เพิ่ม Dataset 'thai-constitutions-pdfs'")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Mount Google Drive  (Colab เท่านั้น — ข้าม Kaggle)
# ═══════════════════════════════════════════════════════════════

if IS_COLAB:
    from google.colab import drive
    print("🔗 กำลัง Mount Google Drive...")
    drive.mount('/content/drive')

    # ตรวจสอบ PDF
    if PDF_DIR.exists():
        pdf_count = len(list(PDF_DIR.glob("*.pdf")))
        print(f"✅ พบ PDF {pdf_count}/38 ไฟล์ใน {PDF_DIR}")
        if pdf_count < 38:
            print(f"⚠️  พบแค่ {pdf_count} ไฟล์")
            print(f"   → อัปโหลด PDF ทั้ง 38 ไฟล์ไปที่ Google Drive: {PDF_DIR}")
    else:
        print(f"❌ ไม่พบ folder: {PDF_DIR}")
        print(f"   → สร้าง folder และอัปโหลด PDF ทั้ง 38 ไฟล์ไปที่ Google Drive")
        PDF_DIR.mkdir(parents=True, exist_ok=True)
        print(f"   → สร้าง folder แล้วที่: {PDF_DIR}")
else:
    print(f"ℹ️  ไม่ใช่ Colab — ข้าม Cell นี้ (Environment: {ENV_NAME})")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Download Pipeline Scripts จาก GitHub
# (ถ้ารันบน Kaggle/Colab — Local จะใช้ script ที่อยู่แล้ว)
# ═══════════════════════════════════════════════════════════════
import urllib.request
import re

PIPELINE_SCRIPTS = [
    "config.py",
    "01_ocr_pipeline.py",
    "02_text_extraction.py",
    "03_validate_output.py",
    "run_pipeline.py",
]

if IS_KAGGLE or IS_COLAB:
    print(f"⬇️  กำลังดาวน์โหลด scripts จาก GitHub...")
    print(f"   Source: {GITHUB_RAW_BASE}/")
    print(f"   Dest  : {SCRIPT_DIR}/")
    print()

    for script in PIPELINE_SCRIPTS:
        url      = f"{GITHUB_RAW_BASE}/{script}"
        dst_path = SCRIPT_DIR / script
        try:
            urllib.request.urlretrieve(url, dst_path)
            size_kb = dst_path.stat().st_size / 1024
            print(f"  ✅ {script:<30} ({size_kb:.1f} KB)")
        except Exception as e:
            print(f"  ❌ {script:<30} — {e}")
            print(f"     → ตรวจสอบ: repo public ไหม? และ branch '{GITHUB_BRANCH}' ถูกต้องไหม?")

else:
    print(f"ℹ️  Local mode — ใช้ scripts ที่อยู่ใน {SCRIPT_DIR} เลย")

print(f"\n📁 Scripts ใน {SCRIPT_DIR}:")
for f in sorted(SCRIPT_DIR.glob("*.py")):
    print(f"   {f.name}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Patch config.py ให้ใช้ PATH ที่ถูกต้องสำหรับ Environment นี้
# ═══════════════════════════════════════════════════════════════
import re

config_path = SCRIPT_DIR / "config.py"

if not config_path.exists():
    raise FileNotFoundError(f"❌ ไม่พบ config.py ที่ {config_path} — รัน Cell 5 ก่อน")

content = config_path.read_text(encoding="utf-8")

# Override directory paths
PATH_OVERRIDES = [
    (r'^BASE_DIR\s*=.*$',       f'BASE_DIR       = Path(r"{SCRIPT_DIR}")'),
    (r'^RAW_PDF_DIR\s*=.*$',    f'RAW_PDF_DIR    = Path(r"{PDF_DIR}")'),
    (r'^OCR_OUTPUT_DIR\s*=.*$', f'OCR_OUTPUT_DIR = Path(r"{OCR_CACHE_DIR}")'),
    (r'^TEXT_OUTPUT_DIR\s*=.*$',f'TEXT_OUTPUT_DIR= Path(r"{TEXT_DIR}")'),
    (r'^PROCESSED_DIR\s*=.*$',  f'PROCESSED_DIR  = Path(r"{PROCESSED_DIR}")'),
]

for pattern, replacement in PATH_OVERRIDES:
    new_content = re.sub(pattern, replacement, content, flags=re.MULTILINE)
    if new_content != content:
        content = new_content
        print(f"  ✅ override: {replacement.strip()}")
    else:
        print(f"  ⚠️  ไม่พบ pattern: {pattern}")

# ลบ mkdir loop (ไม่อยากให้ config สร้าง folder ผิดที่)
content = re.sub(
    r'# Ensure directories exist.*?d\.mkdir\(parents=True, exist_ok=True\)',
    '# Directories managed by runner notebook',
    content, flags=re.DOTALL
)

config_path.write_text(content, encoding="utf-8")
print(f"\n✅ config.py อัปเดตแล้วที่ {config_path}")

# ── Override log file paths ด้วย ────────────────────────────────
# 01_ocr_pipeline.py ใช้ ocr_pipeline.log → เปลี่ยนให้ไปที่ logs/
for script_name, old_log, new_log in [
    ("01_ocr_pipeline.py",   "ocr_pipeline.log",      str(LOG_DIR / "ocr_pipeline.log")),
    ("02_text_extraction.py", "text_extraction.log",  str(LOG_DIR / "text_extraction.log")),
]:
    sp = SCRIPT_DIR / script_name
    if sp.exists():
        sc = sp.read_text(encoding="utf-8")
        sc = sc.replace(f'"{old_log}"', f'r"{new_log}"')
        sp.write_text(sc, encoding="utf-8")
        print(f"  📝 {script_name}: log → {new_log}")

print("\n✅ Patch เสร็จสิ้น")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — API Key Setup
# ═══════════════════════════════════════════════════════════════

api_key = None

if IS_KAGGLE:
    # ── Kaggle Secrets ──────────────────────────────────────────
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("TYPHOON_OCR_API_KEY")
        print("✅ โหลด API Key จาก Kaggle Secrets สำเร็จ")
    except Exception as e:
        print(f"❌ ไม่พบ Kaggle Secret: {e}")
        print("   → ไปที่ Notebook Settings → Secrets → เพิ่ม TYPHOON_OCR_API_KEY")

elif IS_COLAB:
    # ── Google Colab Secrets ─────────────────────────────────────
    try:
        from google.colab import userdata
        api_key = userdata.get("TYPHOON_OCR_API_KEY")
        print("✅ โหลด API Key จาก Colab Secrets สำเร็จ")
    except Exception as e:
        print(f"❌ ไม่พบ Colab Secret: {e}")
        print("   → ไปที่ 🔑 Secrets (ไอคอนกุญแจซ้ายมือ) → เพิ่ม TYPHOON_OCR_API_KEY")

else:
    # ── Local .env ───────────────────────────────────────────────
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("TYPHOON_OCR_API_KEY")
    if api_key:
        print("✅ โหลด API Key จาก .env สำเร็จ")
    else:
        print("❌ ไม่พบ TYPHOON_OCR_API_KEY ใน .env")

# ตั้ง environment variable
if api_key:
    os.environ["TYPHOON_OCR_API_KEY"] = api_key
    masked = api_key[:6] + "*" * (len(api_key) - 10) + api_key[-4:] if len(api_key) > 10 else "***"
    print(f"   Key (masked): {masked}")
else:
    print("\n⚠️  WARNING: ไม่มี API Key — OCR Pipeline จะ fail!")
    print("   (Text Extraction ยังทำงานได้โดยไม่ต้องใช้ API Key)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Verify Setup (ตรวจสอบทุกอย่างก่อนรัน)
# ═══════════════════════════════════════════════════════════════
import json

ok = True
checks = []

# 1. PDF files
pdf_files = list(PDF_DIR.glob("*.pdf")) if PDF_DIR.exists() else []
pdf_ok = len(pdf_files) == 38
checks.append((pdf_ok, f"PDF files: {len(pdf_files)}/38 ในโฟลเดอร์ {PDF_DIR.name}"))
if not pdf_ok: ok = False

# 2. API Key
key_ok = bool(os.environ.get("TYPHOON_OCR_API_KEY"))
checks.append((key_ok, "TYPHOON_OCR_API_KEY: ตั้งค่าแล้ว"))
if not key_ok: ok = False

# 3. Scripts
for sc in ["config.py", "01_ocr_pipeline.py", "02_text_extraction.py"]:
    sc_ok = (SCRIPT_DIR / sc).exists()
    checks.append((sc_ok, f"Script: {sc}"))
    if not sc_ok: ok = False

# 4. Output directories
for d, name in [(OCR_CACHE_DIR, "ocr_cache"), (PROCESSED_DIR, "processed"), (LOG_DIR, "logs")]:
    checks.append((d.exists(), f"Output dir: {name}/"))

# 5. Internet (Kaggle ต้องเปิด)
try:
    import urllib.request
    urllib.request.urlopen("https://api.opentyphoon.ai", timeout=5)
    net_ok = True
except:
    net_ok = False
checks.append((net_ok, "Internet / API endpoint reachable"))
if not net_ok: ok = False

print("\n📋 ผลการตรวจสอบ:\n")
for passed, msg in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {msg}")

print()
if ok:
    print("🚀 ทุกอย่างพร้อม — รัน Cell ถัดไปเพื่อเริ่ม Pipeline!")
else:
    print("⚠️  แก้ไขปัญหาด้านบนก่อน แล้วค่อยรัน Pipeline")

# แสดงจำนวน PDF แยกตามชนิด
if pdf_files:
    img_pdfs = [f for f in pdf_files if int(f.name.split('_')[0]) <= 32]
    txt_pdfs = [f for f in pdf_files if int(f.name.split('_')[0]) > 32]
    print(f"\n  📊 Image PDF (OCR)      : {len(img_pdfs)} ไฟล์ (1–32)")
    print(f"  📄 Text PDF (Extract)   : {len(txt_pdfs)} ไฟล์ (33–38)")

# Resume status
cached = list(OCR_CACHE_DIR.rglob("page_*.md"))
if cached:
    print(f"\n  ♻️  Resume: พบ OCR cache {len(cached)} หน้า — Pipeline จะข้ามหน้าที่ทำแล้ว")

---
## 🔴 STEP 1 — OCR Pipeline (Image PDFs ไฟล์ที่ 1–32)

- ใช้ **Typhoon OCR 1.5 API** ทำ OCR ทีละหน้า
- Rate limit: 20 req/min → Pipeline มี sleep อัตโนมัติ
- มี **cache** ทีละหน้า → ถ้า session ตาย ให้รัน Cell นี้ใหม่ — จะ **resume** ต่อจากที่ค้างไว้
- เวลาโดยประมาณ: **8–12 ชั่วโมง** (ขึ้นกับจำนวนหน้ารวม)

> 💡 **Kaggle Tip:** ถ้า session ใกล้ครบ 9 ชั่วโมง ให้ดาวน์โหลด Output → เซฟ `ocr_cache/` ไว้ก่อน
> แล้ว Session ใหม่ upload `ocr_cache/` กลับมา Pipeline จะ resume ต่อทันที

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Run OCR Pipeline
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, time
from datetime import datetime

if not os.environ.get("TYPHOON_OCR_API_KEY"):
    raise RuntimeError("❌ ไม่มี API Key — กลับไปรัน Cell 7 ก่อน")

print(f"🚀 เริ่ม OCR Pipeline: {datetime.now().strftime('%H:%M:%S')}")
print(f"   Script : {SCRIPT_DIR / '01_ocr_pipeline.py'}")
print(f"   PDF_DIR: {PDF_DIR}")
print(f"   Cache  : {OCR_CACHE_DIR}")
print()

t_start = time.time()

proc = subprocess.Popen(
    [
        sys.executable,
        str(SCRIPT_DIR / "01_ocr_pipeline.py"),
        "--skip-existing",   # resume จาก cache
    ],
    cwd=str(SCRIPT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ},
)

# แสดง output แบบ real-time
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

elapsed = (time.time() - t_start) / 60
if proc.returncode == 0:
    print(f"\n✅ OCR Pipeline เสร็จสิ้น ({elapsed:.1f} นาที)")
else:
    print(f"\n❌ OCR Pipeline จบด้วย error (return code: {proc.returncode})")
    print(f"   ดู log ที่: {LOG_DIR / 'ocr_pipeline.log'}")

# Checkpoint: นับผลลัพธ์
ocr_done = list(PROCESSED_DIR.glob("const_2*.json"))
print(f"   📦 ไฟล์ JSON ที่ได้: {len(ocr_done)} ไฟล์")

---
## 🔵 STEP 2 — Text Extraction (Text PDFs ไฟล์ที่ 33–38)

- ใช้ **PyMuPDF + pdfplumber** ดึงข้อความโดยตรง — **ไม่ต้องใช้ API**
- เร็วมาก: **< 5 นาที** สำหรับทั้ง 6 ไฟล์

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Run Text Extraction Pipeline
# ═══════════════════════════════════════════════════════════════
from datetime import datetime
import time

print(f"🚀 เริ่ม Text Extraction: {datetime.now().strftime('%H:%M:%S')}")
print(f"   Script : {SCRIPT_DIR / '02_text_extraction.py'}")
print(f"   PDF_DIR: {PDF_DIR}")
print()

t_start = time.time()

proc = subprocess.Popen(
    [
        sys.executable,
        str(SCRIPT_DIR / "02_text_extraction.py"),
        "--skip-existing",
        "--method", "smart",
    ],
    cwd=str(SCRIPT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ},
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

elapsed = (time.time() - t_start) / 60
if proc.returncode == 0:
    print(f"\n✅ Text Extraction เสร็จสิ้น ({elapsed:.1f} นาที)")
else:
    print(f"\n❌ Text Extraction จบด้วย error")
    print(f"   ดู log ที่: {LOG_DIR / 'text_extraction.log'}")

# สรุปผล
all_json = list(PROCESSED_DIR.glob("const_*.json"))
print(f"\n📦 สรุป: ได้ JSON ทั้งหมด {len(all_json)}/38 ไฟล์")

---
## ✅ STEP 3 — Validation & QA Report

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — Run Validation
# ═══════════════════════════════════════════════════════════════
print(f"🔍 เริ่ม Validation...")

proc = subprocess.Popen(
    [sys.executable, str(SCRIPT_DIR / "03_validate_output.py")],
    cwd=str(SCRIPT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ},
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

if proc.returncode == 0:
    print("\n✅ Validation เสร็จสิ้น")
else:
    print("\n⚠️  Validation จบด้วย warning/error — ดู output ด้านบน")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — Results Summary
# ═══════════════════════════════════════════════════════════════
import json
import pandas as pd
from IPython.display import display, HTML

print("📊 สรุปผลลัพธ์\n")

json_files = sorted(PROCESSED_DIR.glob("const_*.json"))
rows = []

for jf in json_files:
    try:
        d = json.loads(jf.read_text(encoding="utf-8"))
        rows.append({
            "id":            d.get("id", jf.stem),
            "ปี พ.ศ.": d.get("year_th", ""),
            "ชื่อย่อ":   d.get("name_short", ""),
            "ประเภท":    "🖼️ OCR" if d.get("source_type") == "image_pdf" else "📄 Text",
            "จำนวนหน้า":d.get("total_pages", 0),
            "จำนวนคำ":   d.get("metadata", {}).get("total_words_approx", 0),
            "method":       d.get("processing_method", ""),
        })
    except Exception as e:
        print(f"⚠️  อ่านไม่ได้: {jf.name} — {e}")

if rows:
    df = pd.DataFrame(rows).sort_values("ปี พ.ศ.").reset_index(drop=True)
    print(f"  ✅ ประมวลผลแล้ว : {len(df)} / 38 ฉบับ")
    print(f"  🖼️  Image PDF   : {len(df[df['ประเภท'].str.contains('OCR')])} ฉบับ")
    print(f"  📄  Text PDF    : {len(df[df['ประเภท'].str.contains('Text')])} ฉบับ")
    print(f"  💬  คำรวม      : {df['จำนวนคำ'].sum():,} คำ")
    print()
    display(df.drop(columns=["id", "method"]).style
        .format({"จำนวนคำ": "{:,}"})
        .background_gradient(subset=["จำนวนคำ"], cmap="Blues")
        .set_properties(**{"text-align": "left", "padding": "4px 10px"})
        .hide(axis="index"))
else:
    print("⚠️  ยังไม่มีผลลัพธ์ — ตรวจสอบว่า Pipeline รันสำเร็จ")

---
## 📥 STEP 4 — Package Output สำหรับดาวน์โหลด

### Kaggle
- ไฟล์ทั้งหมดใน `/kaggle/working/processed/` จะถูก zip และดาวน์โหลดได้จาก **Output** tab
- หรือ Commit Notebook เพื่อ save เป็น Kaggle Dataset output

### Colab
- ไฟล์ทั้งหมดอยู่ใน Google Drive แล้ว — ดาวน์โหลดจาก Drive ได้เลย
- หรือ Cell นี้จะ zip และดาวน์โหลดอัตโนมัติ

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13 — Package & Download Output
# ═══════════════════════════════════════════════════════════════
import shutil, zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name  = f"constitution_processed_{timestamp}.zip"

if IS_KAGGLE:
    zip_path = WORK_DIR / zip_name
elif IS_COLAB:
    zip_path = Path("/content") / zip_name
else:
    zip_path = WORK_DIR / zip_name

print(f"📦 กำลัง zip ผลลัพธ์ → {zip_path}")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PROCESSED_DIR.rglob("*")):
        if f.is_file():
            arcname = f"processed/{f.name}"
            zf.write(f, arcname)

zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"✅ Zip เสร็จ: {zip_name} ({zip_size_mb:.1f} MB)")

# นับไฟล์ใน zip
with zipfile.ZipFile(zip_path) as zf:
    print(f"   มีไฟล์ทั้งหมด: {len(zf.namelist())} ไฟล์")

print()
if IS_KAGGLE:
    print(f"📥 Kaggle: ไปที่แถบ 'Output' ด้านขวา → ดาวน์โหลด '{zip_name}'")
    print(f"   หรือ Commit Notebook เพื่อ save เป็น Dataset Output")
elif IS_COLAB:
    print(f"📥 Colab: กำลัง Download...")
    from google.colab import files
    files.download(str(zip_path))
else:
    print(f"📥 Local: ไฟล์อยู่ที่ {zip_path}")